# MegaDescriptor 파인튜닝 — MPDD  (wildlife-tools 공식 레시피)

공식 학습 노트북 `wildlife-tools/baselines/training/MegaDescriptor-T-224.ipynb` 을 MPDD 용으로 옮긴 것.

**공식과 동일하게 유지한 부분**
- `wildlife_tools.data.WildlifeDataset` + `wildlife_tools.train.ArcFaceLoss` + `BasicTrainer`
- ArcFace `margin=0.5`, `scale=64`
- 증강: `RandomResizedCrop` + `RandAugment(2, 20)`
- 옵티마이저: `SGD(lr=1e-3, momentum=0.9)` + `CosineAnnealingLR`
- `batch_size=64`, `accumulation_steps=2` (유효 배치 128)

**MPDD 용으로 바꾼 부분**
- 데이터: MPDD (Market-1501 스타일) → 메타데이터 DataFrame 직접 구성
- 백본: ImageNet Swin 대신 **기존 MegaDescriptor 가중치에서 이어서** 파인튜닝
- `epochs` 100 → 30 (개체 95개뿐이라 그 이상은 과적합)
- `epoch_callback` 으로 매 epoch MPDD query/gallery 평가 + best 저장

> GPU 필요. CPU면 아주 느립니다 (Colab T4 권장).


## 0. 설치

In [5]:
!pip install -q wildlife-tools timm

## 1. 임포트 & 설정

In [ ]:
import os, re
from itertools import chain

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from torch.optim import SGD
from torch.utils.data import DataLoader

import timm
from wildlife_tools.data import WildlifeDataset
from wildlife_tools.train import ArcFaceLoss, BasicTrainer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- 1. MPDD_ROOT 경로 ----
MPDD_ROOT = "./dataset/Multi-pose dog dataset/MPDD/pytorch"          # 안에 train/ val/ gallery/ query/
# ---- 2. 보호소 동물 경로 ----
SHELTER_ROOT = "processed_animals"  # 개체당 폴더 (최종 평가용). 없으면 None

# ---- 하이퍼파라미터 (공식값 + MPDD 조정) ----
BACKBONE  = "hf-hub:BVRA/MegaDescriptor-B-224"   # 기존 MegaDescriptor 이어서
# BACKBONE = "swin_base_patch4_window7_224"      # ImageNet 부터 (공식 노트북 방식)
EPOCHS    = 30       # 학습데이터 EPOCHS번 반복해서 학습
BATCH     = 64       # 한 번에 GPU에 넣는 이미지 수
ACCUM     = 2        # gradient update는 BATCH * ACCUM장을 본 뒤 한 번 한다.
LR        = 1e-3     # Learning Rate: 학습할 때 가중치를 얼마나 크게 변경할지 결정
IMG_SIZE  = 224      # 모델에 넣을 이미지 크기
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225) # 정규화할 때 사용할 값
CKPT      = "megadescriptor_mpdd_best.pth" # 학습하면서 가장 성능이 좋았던 모델을 megadescriptor_mpdd_best.pth라는 이름으로 저장

# train 폴더가 진짜 존재하는지 검사
if not os.path.isdir(f"{MPDD_ROOT}/train"):
    raise AssertionError(f"train 폴더 없음: {MPDD_ROOT}/train")
print("DEVICE:", DEVICE, "| BACKBONE:", BACKBONE)


DEVICE: cpu | BACKBONE: hf-hub:BVRA/MegaDescriptor-B-224


## 2. MPDD 메타데이터 구성

Market-1501 파일명 `<id>_c<pose>s<seq>_<n>.jpg` 에서 identity / camera 를 뽑아 DataFrame 으로.

In [ ]:
_ID = re.compile(r"^(\d+)_c(\d+)s\d+")

def market_meta(split):
    d = f"{MPDD_ROOT}/{split}" # ex) {MPDD_ROOT}/train
    rows = []
    for f in sorted(os.listdir(d)):
        m = _ID.match(f) # 앞서 만든 정규표현식의 규칙과 매칭되는지 확인
        if m and f.lower().endswith((".jpg", ".jpeg", ".png")): # 1) 원하는 패턴인지? 2) 이미지 파일인지 확인
            rows.append({"path": f"{split}/{f}", "identity": m.group(1),
                         "camera": int(m.group(2)), "split": split})
    return pd.DataFrame(rows)

meta_train = market_meta("train")   # 파인튜닝에 사용할 사진
meta_gal   = market_meta("gallery") # 기준으로 등록해놓은 사진
meta_qry   = market_meta("query")   # 누구인지 찾아야 하는 사진

print(f"train   {len(meta_train):4d}장 / {meta_train['identity'].nunique()}개체")
print(f"gallery {len(meta_gal):4d}장 / {meta_gal['identity'].nunique()}개체")
print(f"query   {len(meta_qry):4d}장 / {meta_qry['identity'].nunique()}개체")
print("train ∩ gallery 개체:",
      len(set(meta_train.identity) & set(meta_gal.identity)), "(0 = open-set, 정상)")


train    921장 / 95개체
gallery  521장 / 96개체
query    103장 / 95개체
train ∩ gallery 개체: 0 (0 = open-set, 정상)


## 3. 데이터셋 & 증강  *(공식과 동일)*

In [ ]:
train_tf = T.Compose([
    T.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0)), # 원본 면적의 80~100% 정도를 사용하는 범위에서 crop을 선택
    T.RandAugment(num_ops=2, magnitude=20), # 여러 이미지 증강 방법 중 랜덤하게 몇 개를 선택해서 적용
    T.ToTensor(), # 이미지를 PyTorch Tensor로 변환
    T.Normalize(mean=MEAN, std=STD), # 각 RGB 채널 정규화
])
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD),
])

train_ds = WildlifeDataset(metadata=meta_train, root=MPDD_ROOT, transform=train_tf)
print("num_classes:", train_ds.num_classes)


num_classes: 95


## 4. 백본 & ArcFace  *(공식과 동일)*

In [12]:
backbone = timm.create_model(BACKBONE, num_classes=0, pretrained=True)

with torch.no_grad():
    embedding_size = backbone(torch.randn(1, 3, IMG_SIZE, IMG_SIZE)).shape[1]

objective = ArcFaceLoss(
    num_classes=train_ds.num_classes,
    embedding_size=embedding_size,
    margin=0.5,
    scale=64,
)
print("embedding_size:", embedding_size)


embedding_size: 1024


## 5. 옵티마이저 & 스케줄러  *(공식과 동일, T_max만 EPOCHS)*

In [ ]:
params = chain(backbone.parameters(), objective.parameters())
optimizer = SGD(params=params, lr=LR, momentum=0.9)

min_lr = LR * 1e-3
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=min_lr)

## 6. 평가 함수 — MPDD query / gallery

re-ID 표준: 각 query 를 gallery 와 코사인 유사도로 랭킹.
Market 규칙대로 **같은 개체 & 같은 pose** 는 junk 로 제외. → mAP, Top-1, Top-5.

In [14]:
@torch.no_grad()
def embed(model, meta, root):
    ds = WildlifeDataset(metadata=meta, root=root, transform=eval_tf, load_label=False)
    dl = DataLoader(ds, batch_size=128, num_workers=2)
    model.eval()
    out = [F.normalize(model(x.to(DEVICE))).cpu() for x in dl]
    return torch.cat(out).numpy()


def _ap_cmc(rel):
    if not rel.any():
        return 0.0, 0, 0
    hits = np.cumsum(rel)
    ranks = np.arange(1, len(rel) + 1)
    return float((hits / ranks * rel).sum() / rel.sum()), int(rel[0]), int(rel[:5].any())


def evaluate_qg(q_emb, q_id, q_cam, g_emb, g_id, g_cam):
    q_id, g_id = np.asarray(q_id), np.asarray(g_id)
    q_cam, g_cam = np.asarray(q_cam), np.asarray(g_cam)
    sim = q_emb @ g_emb.T
    aps = c1 = c5 = 0.0
    for i in range(len(q_id)):
        order = np.argsort(sim[i])[::-1]
        junk = (g_id == q_id[i]) & (g_cam == q_cam[i])
        order = order[~junk[order]]
        ap, h1, h5 = _ap_cmc(g_id[order] == q_id[i])
        aps += ap; c1 += h1; c5 += h5
    n = len(q_id)
    return aps / n, c1 / n, c5 / n


def mpdd_score(model):
    qe = embed(model, meta_qry, MPDD_ROOT)
    ge = embed(model, meta_gal, MPDD_ROOT)
    return evaluate_qg(qe, meta_qry.identity, meta_qry.camera,
                       ge, meta_gal.identity, meta_gal.camera)


## 7. epoch 콜백 — 매 epoch MPDD 평가 + best 저장

In [15]:
best = {"mAP": -1.0}

def on_epoch(trainer, epoch_data):
    mAP, t1, t5 = mpdd_score(trainer.model)
    print(f"  [epoch {trainer.epoch}] loss {epoch_data['train_loss_epoch_avg']:.3f}"
          f"  |  MPDD  mAP {mAP:.4f}  Top-1 {t1:.4f}  Top-5 {t5:.4f}")
    if mAP > best["mAP"]:
        best["mAP"] = mAP
        trainer.save(".", CKPT)
        print(f"      -> best 저장 (mAP {mAP:.4f})  {CKPT}")


## 8. 학습  *(BasicTrainer — 공식과 동일 구조)*

In [16]:
trainer = BasicTrainer(
    dataset=train_ds,
    model=backbone,
    objective=objective,
    optimizer=optimizer,
    scheduler=scheduler,
    batch_size=BATCH,
    accumulation_steps=ACCUM,
    num_workers=2,          # 로컬 Windows 면 0
    epochs=EPOCHS,
    device=DEVICE,
    epoch_callback=on_epoch,
)

print("zero-shot(학습 전):  MPDD  mAP {:.4f}  Top-1 {:.4f}  Top-5 {:.4f}".format(*mpdd_score(backbone)))
trainer.train()
print(f"\n최고 MPDD mAP: {best['mAP']:.4f}  (저장: {CKPT})")


zero-shot(학습 전):  MPDD  mAP 0.6939  Top-1 0.8447  Top-5 0.9417


Epoch 0:  13%|███████▎                                               | 2/15 [01:26<09:20, 43.14s/it]


KeyboardInterrupt: 

## 9. 최종 평가 — 보호소 데이터(`processed_animals`)

MPDD 숫자는 논문 비교용. **실제 제품 지표는 여기.**
best 체크포인트를 백본에 로드해 개체당 절반 gallery / 절반 query 로 mAP·Top-1 (5시드 평균).

In [ ]:
def df_from_folder(root):
    rows = []
    for d in sorted(os.listdir(root)):
        dd = os.path.join(root, d)
        if os.path.isdir(dd):
            for f in sorted(os.listdir(dd)):
                if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                    rows.append({"path": f"{d}/{f}", "identity": d})
    return pd.DataFrame(rows)


def evaluate_map_split(emb, ids, gallery_frac=0.5, seed=0):
    ids = np.asarray(ids)
    rng = np.random.default_rng(seed)
    g_idx, q_idx = [], []
    for c in np.unique(ids):
        idx = np.where(ids == c)[0].copy(); rng.shuffle(idx)
        if len(idx) <= 1:
            g_idx += idx.tolist(); continue
        k = max(1, round(len(idx) * gallery_frac))
        g_idx += idx[:k].tolist(); q_idx += idx[k:].tolist()
    g_idx, q_idx = np.array(g_idx), np.array(q_idx)
    sim = emb[q_idx] @ emb[g_idx].T
    g_id = ids[g_idx]
    aps = c1 = c5 = 0.0
    for row, gt in zip(sim, ids[q_idx]):
        ap, h1, h5 = _ap_cmc(g_id[np.argsort(row)[::-1]] == gt)
        aps += ap; c1 += h1; c5 += h5
    q = len(q_idx)
    return aps / q, c1 / q, c5 / q


if SHELTER_ROOT and os.path.isdir(SHELTER_ROOT):
    ck = torch.load(CKPT, map_location=DEVICE)
    backbone.load_state_dict(ck["model"]); backbone.eval()

    meta_sh = df_from_folder(SHELTER_ROOT)
    emb_sh = embed(backbone, meta_sh, SHELTER_ROOT)
    ids_sh = meta_sh["identity"].to_numpy()

    r = np.array([evaluate_map_split(emb_sh, ids_sh, seed=s) for s in range(5)])
    m, sd = r.mean(0), r.std(0)
    print(f"[finetuned / processed_animals]  개체 {meta_sh.identity.nunique()} / 이미지 {len(meta_sh)}")
    print(f"  split  mAP {m[0]:.4f}±{sd[0]:.4f} | Top-1 {m[1]:.4f}±{sd[1]:.4f} | Top-5 {m[2]:.4f}±{sd[2]:.4f}")
else:
    print("SHELTER_ROOT 없음 - MPDD 결과만 사용")


---
### 참고

- **전체 백본 파인튜닝**이 기본 (공식 `chain(backbone.parameters(), objective.parameters())`). 95개체로 과적합 조짐 보이면
  뒤쪽 stage만 학습하도록 셀 5 앞에 넣기:
  ```python
  for p in backbone.parameters(): p.requires_grad = False
  for p in backbone.layers[-1].parameters(): p.requires_grad = True
  for p in getattr(backbone, "norm", []).parameters(): p.requires_grad = True
  params = chain((p for p in backbone.parameters() if p.requires_grad), objective.parameters())
  ```
- 로컬 Windows/Jupyter 에서 돌리면 `num_workers=2` → `0` (셀 8, 셀 6).
- `BACKBONE` 을 `-L-384` 로 바꾸면 `BATCH=16`, `IMG_SIZE=384`.
